# Exploratory Data Analysis (EDA)
## Public Compliance Data Analysis - MBA Thesis

**Objective:** Explore the Gold layer datasets at **municipality level** (5,570 records, 1 row per Brazilian municipality) to understand:
- Data distributions and summary statistics
- Missing values and data quality
- Initial patterns and relationships
- Regional variations in compliance and socioeconomic indicators

**Grain note.** The main analytical dataset used here is `analysis_compliance_municipality` (one row per municipality, N=5,570). The previous state-level rollup (`analysis_compliance`, N=27) had too few observations for meaningful correlation/regression analysis. To preserve geographic context, `state_code` / `state_name` / `region_code` / `region_name` are kept as identifier columns and one-hot encoded into dummy variables (`is_region_*`, `is_state_*`) the same way the state-level dataset encoded region.

In [ ]:
# --- AUTO-GENERATED DEPENDENCY INSTALL ---
# Installs all project dependencies on first run (Colab, fresh environments, etc).
# Idempotent: pip skips anything already installed.
# To regenerate this cell, run: python scripts/inject_pip_install.py

import subprocess
import sys
from pathlib import Path

_req = Path.cwd().parent / "requirements.txt"
if not _req.exists():
    _req = Path.cwd() / "requirements.txt"

if _req.exists():
    print(f"Installing dependencies from {_req.name if _req.exists() else "requirements.txt"} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(_req)])
    print("Dependencies ready.")
else:
    print("requirements.txt not found. Install manually: pip install -r requirements.txt")


## Step-by-step

1. Packages and environment setup
2. Reproducibility
3. Data loading
4. Analysis blocks
5. Summary and interpretation


# Packages


In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Try local loader first, fallback to S3 if available
from src.analysis.local_data_loader import LocalGoldDataLoader as GoldDataLoader

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)
pd.set_option('display.float_format', '{:.2f}'.format)


In [ ]:
import matplotlib as mpl
mpl.rcParams['axes.formatter.useoffset'] = False
mpl.rcParams['axes.formatter.limits'] = (-99, 99)


# Reproducibility


In [ ]:
import os
import random

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

print(f"Reproducibility seed fixed at {SEED}")


In [ ]:
import json as _json

_rtcfg_path = os.path.join('..', 'config', 'runtime_config.json')
if os.path.exists(_rtcfg_path):
    with open(_rtcfg_path) as _f:
        _rtcfg = _json.load(_f)
else:
    _rtcfg = {}

S3_BUCKET_NAME = os.environ.get('S3_BUCKET_NAME', _rtcfg.get('aws', {}).get('s3_bucket_name', ''))
AWS_PROFILE = os.environ.get('AWS_PROFILE', _rtcfg.get('aws', {}).get('profile', None))


## 1. Load Data

In [ ]:
loader = GoldDataLoader()

print("Available datasets:")
for dataset in loader.list_available_datasets():
    print(f"  • {dataset}")


In [ ]:
datasets = loader.load_all()

df_muni = datasets.get('municipality_socioeconomic')
df_state = datasets.get('state_summary')
df_sanctions = datasets.get('sanctions_summary')

# Primary analytical dataset: one row per municipality (N ~= 5,570).
df_analysis = datasets.get('analysis_compliance_municipality')

# --- Add one-hot region and state dummies (not present in the muni dataset). ---
# These mirror the is_norte / is_nordeste / ... columns that existed on the
# state-level dataset -- keeping the same human-readable names so any downstream
# notebook that referenced those columns continues to work.
# Kept as Int64 (0/1) for consistency with the pre-existing is_* convention and
# for direct use as regression features in notebooks 02 / 03.
REGION_NAME_TO_DUMMY = {
    'Norte': 'is_norte',
    'Nordeste': 'is_nordeste',
    'Sudeste': 'is_sudeste',
    'Sul': 'is_sul',
    'Centro-Oeste': 'is_centro_oeste',
}
for rname, col in REGION_NAME_TO_DUMMY.items():
    df_analysis[col] = (df_analysis['region_name'] == rname).astype('Int64')
REGION_DUMMY_COLS = list(REGION_NAME_TO_DUMMY.values())

# State dummies: one per state, named by 2-digit IBGE state code.
state_dummies = pd.get_dummies(df_analysis['state_code'], prefix='is_state').astype('Int64')
df_analysis = pd.concat([df_analysis, state_dummies], axis=1)
STATE_DUMMY_COLS = list(state_dummies.columns)

print(f"\n✅ Loaded {len(datasets)} datasets")
print(f"   Main analytical dataset: analysis_compliance_municipality")
print(f"   Rows: {len(df_analysis):,} municipalities, {df_analysis['state_code'].nunique()} states, {df_analysis['region_code'].nunique()} regions")
print(f"   Region dummies added ({len(REGION_DUMMY_COLS)}): {REGION_DUMMY_COLS}")
print(f"   State dummies added ({len(STATE_DUMMY_COLS)}): first 3 = {STATE_DUMMY_COLS[:3]} ... last = {STATE_DUMMY_COLS[-1]}")

## 2. Dataset Overview

In [ ]:
print("=" * 80)
print("ANALYSIS COMPLIANCE MUNICIPALITY DATASET")
print("=" * 80)
print(f"Shape: {df_analysis.shape}  (rows = municipalities, cols = features + dummies)")
print(f"\nDtypes (non-dummy columns only):")
print(df_analysis.drop(columns=REGION_DUMMY_COLS + STATE_DUMMY_COLS).dtypes)
print(f"\nMemory Usage: {df_analysis.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

In [ ]:
# Preview a handful of columns (hiding the 32 region+state dummies).
preview_cols = [c for c in df_analysis.columns if c not in REGION_DUMMY_COLS + STATE_DUMMY_COLS]
df_analysis[preview_cols].head(10)

## 2.1 Complete Gold Layer Dataset Inventory

This section provides a comprehensive overview of **ALL 6 Gold layer datasets**.

| Dataset | Rows | Grain | Purpose |
|---------|------|-------|---------|
| `municipality_socioeconomic` | 5,570 | Municipality | 2010→2022 census comparison with inflation-adjusted income |
| `state_summary` | 27 | State | State-level aggregations (pop-weighted) |
| `sanctions_summary` | 3 | Registry type | Sanctions by CEIS/CNEP/CEPIM registries |
| `analysis_compliance` | 27 | State | State-level compliance analysis (legacy) |
| `analysis_compliance_municipality` | 5,570 | Municipality | **Primary analysis dataset** - main analytical grain |
| `consolidated_clustering` | 5,565 | Municipality | ML-ready with z-score normalized features |

### Key Income Fields Explained

**Inflation-adjusted income columns (IPCA-adjusted to 2022 BRL):**
- `avg_income_real_2022_2022_brl` - 2022 income in 2022 BRL (base year)
- `avg_income_real_2010_2022_brl` - 2010 income restated to 2022 BRL
- `income_change_real_pct` - Real income change % (2010→2022), inflation-adjusted

**Log-transformed columns (for regression analysis):**
- `log_population` - Natural log of population (handles skewness)
- `log_income` - Natural log of real income (base: avg_income_real_2022_2022_brl)
- `log_total_transfers` - Log1p of federal transfers (handles zeros)


## Gold Dataset Field Documentation
Detailed documentation for all 6 Gold layer datasets, including field types, descriptions, calculations, and rationale.

### Dataset: municipality_socioeconomic
**Municipality-level socioeconomic aggregation with 2010→2022 census comparison and inflation-adjusted income metrics**
| Field Name | Type | Description | Calculation Method | Rationale |
|------------|------|-------------|-------------------|-----------|
| `municipality_code` | string | Unique 7-digit IBGE municipality identifier | From dim_municipalities | Stable identifier for joining across datasets |
| `municipality_name` | string | Official municipality name | From dim_municipalities | Human-readable identifier |
| `state_code` | string | 2-digit IBGE state code | From dim_municipalities | State aggregation key |
| `state_name` | string | Full state name | From dim_municipalities | Human-readable state name |
| `region_code` | int | 1-digit IBGE region code (1-5) | From dim_municipalities | Macro-region aggregation |
| `region_name` | string | Region name (Norte, Nordeste, etc.) | From dim_municipalities | Human-readable region |
| `population_2010` | int | Total population in 2010 census | From fact_population where year=2010 | Baseline for population change analysis |
| `population_2022` | int | Total population in 2022 census | From fact_population where year=2022 | Current population for comparison |
| `population_change_pct` | float | Population change % (2010→2022) | ((pop_2022 - pop_2010) / pop_2010) * 100 | Demographic growth indicator |
| `literacy_rate_2010` | float | Literacy rate % in 2010 | From fact_literacy where year=2010 | Baseline education level |
| `literacy_rate_2022` | float | Literacy rate % in 2022 | From fact_literacy where year=2022 | Current education level |
| `literacy_change_pp` | float | Literacy change in percentage points | literacy_2022 - literacy_2010 | Education improvement metric |
| `avg_income_2010` | float | Nominal average income in 2010 (BRL) | From fact_income where year=2010 | Raw income baseline |
| `avg_income_2022` | float | Nominal average income in 2022 (BRL) | From fact_income where year=2022 | Raw current income |
| `avg_income_real_2010_2022_brl` | float | 2010 income adjusted to 2022 BRL using IPCA | From fact_income.avg_income_real_2022_brl where year=2010 | Inflation-adjusted baseline for real comparison |
| `avg_income_real_2022_2022_brl` | float | 2022 income in 2022 BRL (base year) | From fact_income.avg_income_real_2022_brl where year=2022 | Real income in base year currency |
| `income_change_pct` | float | Nominal income change % | ((income_2022 - income_2010) / income_2010) * 100 | Raw income growth |
| `income_change_real_pct` | float | Real income change % (inflation-adjusted) | ((income_real_2022 - income_real_2010) / income_real_2010) * 100 | True purchasing power change |
| `households_2010` | int | Total households in 2010 | From fact_sanitation where year=2010 | Household baseline |
| `households_2022` | int | Total households in 2022 | From fact_sanitation where year=2022 | Current household count |
| `households_change_pct` | float | Household change % (2010→2022) | ((hh_2022 - hh_2010) / hh_2010) * 100 | Household growth indicator |

### Dataset: state_summary
**State-level aggregations with population-weighted metrics and sanctions summary**
| Field Name | Type | Description | Calculation Method | Rationale |
|------------|------|-------------|-------------------|-----------|
| `state_code` | string | 2-digit IBGE state code | From dim_municipalities | State identifier |
| `state_name` | string | Full state name | From dim_municipalities | Human-readable state name |
| `region_code` | int | 1-digit IBGE region code | From dim_municipalities | Region aggregation |
| `region_name` | string | Region name | From dim_municipalities | Human-readable region |
| `n_municipalities` | int | Count of municipalities in state | COUNT DISTINCT municipality_code | State size indicator |
| `population` | int | Total state population 2022 | SUM of municipality populations | State population for weighting |
| `avg_literacy_rate` | float | State average literacy rate | MEAN of municipality literacy rates | Education level indicator |
| `avg_income` | float | State average income (BRL) | MEAN of municipality incomes | Economic level indicator |
| `n_sanctions` | int | Total sanctions in state | COUNT DISTINCT sanction_id | Compliance risk indicator |
| `n_sanctions_ceis` | int | Sanctions from CEIS registry | COUNT WHERE registry_type='CEIS' | Federal contract sanctions |
| `n_sanctions_cnep` | int | Sanctions from CNEP registry | COUNT WHERE registry_type='CNEP' | National penalties registry |
| `n_sanctions_cepim` | int | Sanctions from CEPIM registry | COUNT WHERE registry_type='CEPIM' | Municipal irregularities |
| `sanctions_per_100k` | float | Sanctions per 100,000 population | (n_sanctions / population) * 100000 | Normalized compliance risk |
| `log_population` | float | Natural log of population | LN(population) | For regression normality |
| `log_income` | float | Natural log of income | LN(avg_income) | For regression normality |
| `region_norte` | int | Dummy: 1 if in Norte region | 1 IF region_code=1 ELSE 0 | Regression control variable |
| `region_nordeste` | int | Dummy: 1 if in Nordeste region | 1 IF region_code=2 ELSE 0 | Regression control variable |
| `region_sudeste` | int | Dummy: 1 if in Sudeste region | 1 IF region_code=3 ELSE 0 | Regression control variable |
| `region_sul` | int | Dummy: 1 if in Sul region | 1 IF region_code=4 ELSE 0 | Regression control variable |
| `region_centro_oeste` | int | Dummy: 1 if in Centro-Oeste | 1 IF region_code=5 ELSE 0 | Regression control variable |

### Dataset: sanctions_summary
**Aggregated sanctions view by registry type (CEIS, CNEP, CEPIM)**
| Field Name | Type | Description | Calculation Method | Rationale |
|------------|------|-------------|-------------------|-----------|
| `registry_type` | string | Sanction registry type | From fact_sanctions | Categorization key |
| `n_sanctions` | int | Count of sanctions in registry | COUNT DISTINCT sanction_id | Volume indicator |
| `n_municipalities` | int | Municipalities with sanctions | COUNT DISTINCT municipality_code | Geographic spread |
| `n_states` | int | States with sanctions in registry | COUNT DISTINCT state_code | State coverage |
| `avg_sanctions_per_muni` | float | Average sanctions per municipality | n_sanctions / n_municipalities | Intensity measure |
| `pct_of_total` | float | Percentage of total sanctions | (registry_count / total) * 100 | Relative importance |

### Dataset: analysis_compliance
**State-level compliance analysis dataset (legacy) with region dummy variables**
| Field Name | Type | Description | Calculation Method | Rationale |
|------------|------|-------------|-------------------|-----------|
| `state_code` | string | 2-digit IBGE state code | From dim_municipalities | State identifier |
| `state_name` | string | Full state name | From dim_municipalities | Human-readable state name |
| `region_code` | int | 1-digit IBGE region code | From dim_municipalities | Region aggregation |
| `region_name` | string | Region name | From dim_municipalities | Human-readable region |
| `n_municipalities` | int | Count of municipalities | COUNT DISTINCT municipality_code | State size |
| `population` | int | State population 2022 | SUM of municipality populations | Population weight |
| `avg_literacy_rate` | float | State average literacy rate | MEAN of municipality literacy | Education control |
| `avg_income` | float | State average income | MEAN of municipality income | Economic control |
| `n_sanctions` | int | Total sanctions | COUNT DISTINCT sanction_id | Treatment variable |
| `n_sanctions_ceis` | int | CEIS sanctions | COUNT WHERE registry_type='CEIS' | Federal contract violations |
| `n_sanctions_cnep` | int | CNEP sanctions | COUNT WHERE registry_type='CNEP' | National penalties |
| `n_sanctions_cepim` | int | CEPIM sanctions | COUNT WHERE registry_type='CEPIM' | Municipal irregularities |
| `sanctions_per_100k` | float | Sanctions per 100k population | (n_sanctions / population) * 100000 | Normalized treatment |
| `log_population` | float | Natural log of population | LN(population) | Control for skewness |
| `log_income` | float | Natural log of income | LN(avg_income) | Control for skewness |
| `has_sanctions` | int | Binary: has any sanctions | 1 IF n_sanctions > 0 ELSE 0 | Binary treatment indicator |

### Dataset: analysis_compliance_municipality
**Primary analysis dataset at municipality grain - main analytical dataset for compliance research**
| Field Name | Type | Description | Calculation Method | Rationale |
|------------|------|-------------|-------------------|-----------|
| `municipality_code` | string | Unique 7-digit IBGE identifier | From dim_municipalities | Primary key |
| `municipality_name` | string | Official municipality name | From dim_municipalities | Human-readable name |
| `state_code` | string | 2-digit IBGE state code | From dim_municipalities | State aggregation |
| `state_name` | string | Full state name | From dim_municipalities | Human-readable state |
| `region_code` | int | 1-digit IBGE region code | From dim_municipalities | Region aggregation |
| `region_name` | string | Region name | From dim_municipalities | Human-readable region |
| `population` | int | Population 2022 | From fact_population | Size indicator/control |
| `avg_literacy_rate` | float | Literacy rate % 2022 | From fact_literacy | Human capital control |
| `avg_income` | float | Average income 2022 (BRL) | From fact_income | Economic level control |
| `avg_income_real_2022_2022_brl` | float | Real income in 2022 BRL | From fact_income | Inflation-adjusted economic level |
| `n_sanctions` | int | Total sanctions count | COUNT DISTINCT sanction_id | Primary treatment variable |
| `n_sanctions_ceis` | int | CEIS sanctions count | COUNT WHERE registry_type='CEIS' | Federal contract treatment |
| `n_sanctions_cnep` | int | CNEP sanctions count | COUNT WHERE registry_type='CNEP' | National penalty treatment |
| `n_sanctions_cepim` | int | CEPIM sanctions count | COUNT WHERE registry_type='CEPIM' | Municipal irregularity treatment |
| `sanctions_per_100k` | float | Sanctions per 100k population | (n_sanctions / population) * 100000 | Normalized treatment variable |
| `log_population` | float | Natural log of population | LN(population) | Normality transformation |
| `log_income` | float | Natural log of real income | LN(avg_income_real_2022_2022_brl) | Normality transformation |
| `has_sanctions` | int | Binary: has any sanctions | 1 IF n_sanctions > 0 ELSE 0 | Binary treatment |
| `total_transfers_2022` | float | Total federal transfers 2022 (BRL) | SUM of federal_transfers | Spending treatment variable |
| `transfers_per_capita` | float | Federal transfers per capita | total_transfers / population | Normalized spending |
| `log_total_transfers` | float | Log1p of transfers | LOG1P(total_transfers) | Normality transformation for zeros |

### Dataset: consolidated_clustering
**ML-ready clustering dataset with z-score normalized features and complete case analysis**
| Field Name | Type | Description | Calculation Method | Rationale |
|------------|------|-------------|-------------------|-----------|
| `municipality_code` | string | Unique 7-digit IBGE identifier | From dim_municipalities | Primary key |
| `municipality_name` | string | Official municipality name | From dim_municipalities | Human-readable name |
| `state_code` | string | 2-digit IBGE state code | From dim_municipalities | State identifier |
| `state_name` | string | Full state name | From dim_municipalities | Human-readable state |
| `region_code` | int | 1-digit IBGE region code | From dim_municipalities | Region identifier |
| `region_name` | string | Region name | From dim_municipalities | Human-readable region |
| `population_2010` | int | Population 2010 | From fact_population | Clustering feature |
| `population_2022` | int | Population 2022 | From fact_population | Clustering feature |
| `population_change_pct` | float | Population change % | Calculated | Clustering feature - growth |
| `literacy_rate_2010` | float | Literacy rate 2010 | From fact_literacy | Clustering feature |
| `literacy_rate_2022` | float | Literacy rate 2022 | From fact_literacy | Clustering feature |
| `literacy_change_pp` | float | Literacy change (pp) | Calculated | Clustering feature - improvement |
| `avg_income_real_2010_2022_brl` | float | Real income 2010 | From fact_income | Clustering feature |
| `avg_income_real_2022_2022_brl` | float | Real income 2022 | From fact_income | Clustering feature |
| `income_change_real_pct` | float | Real income change % | Calculated | Clustering feature - growth |
| `households_2010` | int | Households 2010 | From fact_sanitation | Clustering feature |
| `households_2022` | int | Households 2022 | From fact_sanitation | Clustering feature |
| `households_change_pct` | float | Household change % | Calculated | Clustering feature - growth |
| `population_2010_norm` | float | Z-score normalized | (value - mean) / std | ML-ready standardized feature |
| `population_2022_norm` | float | Z-score normalized | (value - mean) / std | ML-ready standardized feature |
| `population_change_pct_norm` | float | Z-score normalized | (value - mean) / std | ML-ready standardized feature |
| `literacy_rate_2010_norm` | float | Z-score normalized | (value - mean) / std | ML-ready standardized feature |
| `literacy_rate_2022_norm` | float | Z-score normalized | (value - mean) / std | ML-ready standardized feature |
| `literacy_change_pp_norm` | float | Z-score normalized | (value - mean) / std | ML-ready standardized feature |
| `avg_income_real_2010_2022_brl_norm` | float | Z-score normalized | (value - mean) / std | ML-ready standardized feature |
| `avg_income_real_2022_2022_brl_norm` | float | Z-score normalized | (value - mean) / std | ML-ready standardized feature |
| `income_change_real_pct_norm` | float | Z-score normalized | (value - mean) / std | ML-ready standardized feature |
| `households_2010_norm` | float | Z-score normalized | (value - mean) / std | ML-ready standardized feature |
| `households_2022_norm` | float | Z-score normalized | (value - mean) / std | ML-ready standardized feature |
| `households_change_pct_norm` | float | Z-score normalized | (value - mean) / std | ML-ready standardized feature |
| `log_population_2022` | float | Natural log of population | LN(population_2022) | Log-transform for skewness |
| `log_avg_income_2022` | float | Natural log of income | LN(avg_income_2022) | Log-transform for skewness |
| `log_households_2022` | float | Natural log of households | LN(households_2022) | Log-transform for skewness |

In [ ]:
# ============================================================
# DATASET 1: municipality_socioeconomic
# Grain: 1 row per municipality (5,570 rows)
# Contains: BOTH 2010 AND 2022 census years with change metrics
# ============================================================
print("=" * 80)
print("DATASET 1: municipality_socioeconomic (Both Census Years)")
print("=" * 80)
print(f"Shape: {df_muni.shape}")
print(f"\nColumns ({len(df_muni.columns)} total):")
print(df_muni.dtypes.to_string())
print(f"\n--- Income-related columns (inflation-adjusted) ---")
income_cols = [c for c in df_muni.columns if 'income' in c.lower()]
for col in income_cols:
    print(f"  • {col}")
print(f"\n--- Sample data (first 5 rows, key columns) ---")
key_cols = ['municipality_code', 'municipality_name', 'population_2010', 'population_2022', 
            'avg_income_2010', 'avg_income_2022', 'avg_income_real_2010_2022_brl', 
            'avg_income_real_2022_2022_brl', 'income_change_real_pct']
display(df_muni[key_cols].head())

In [ ]:
# ============================================================
# DATASET 2: state_summary
# Grain: 1 row per state (27 rows = 26 states + DF)
# Contains: Population-weighted state aggregations
# ============================================================
print("=" * 80)
print("DATASET 2: state_summary (State-level Aggregations)")
print("=" * 80)
print(f"Shape: {df_state.shape}")
print(f"\nColumns ({len(df_state.columns)} total):")
print(df_state.dtypes.to_string())
print(f"\n--- Income columns (both census years) ---")
income_cols = [c for c in df_state.columns if 'income' in c.lower()]
for col in income_cols:
    print(f"  • {col}")
print(f"\n--- Sample data (first 5 states) ---")
display(df_state.head())

In [ ]:
# ============================================================
# DATASET 3: sanctions_summary
# Grain: 1 row per registry type (3 rows: CEIS, CNEP, CEPIM)
# Contains: Sanction counts by registry with PJ/PF breakdown
# ============================================================
print("=" * 80)
print("DATASET 3: sanctions_summary (By Registry Type)")
print("=" * 80)
print(f"Shape: {df_sanctions.shape}")
print(f"\nAll columns:")
print(df_sanctions.dtypes.to_string())
print(f"\n--- Full dataset (all 3 rows) ---")
display(df_sanctions)

In [ ]:
# ============================================================
# DATASET 4: analysis_compliance (State-level)
# Grain: 1 row per state (27 rows)
# Contains: Legacy state-level analysis with log-transformed vars
# Note: Uses 2022 data only (single census year)
# ============================================================
df_analysis_state = datasets.get('analysis_compliance')

print("=" * 80)
print("DATASET 4: analysis_compliance (State-level, Legacy)")
print("=" * 80)
print(f"Shape: {df_analysis_state.shape}")
print(f"\nColumns ({len(df_analysis_state.columns)} total):")
print(df_analysis_state.dtypes.to_string())
print(f"\n--- Log-transformed columns ---")
log_cols = [c for c in df_analysis_state.columns if c.startswith('log_')]
for col in log_cols:
    print(f"  • {col}")
print(f"\n--- Sample data (first 5 states) ---")
display(df_analysis_state.head())

In [ ]:
# ============================================================
# DATASET 5: analysis_compliance_municipality
# Grain: 1 row per municipality (5,570 rows)
# Contains: PRIMARY ANALYTICAL DATASET (main grain)
# Note: Uses 2022 data only + federal transfers
# ============================================================
print("=" * 80)
print("DATASET 5: analysis_compliance_municipality (PRIMARY)")
print("=" * 80)
print(f"Shape: {df_analysis.shape} (includes {len(REGION_DUMMY_COLS)} region + {len(STATE_DUMMY_COLS)} state dummies)")

# Show base columns (excluding dummies)
base_cols = [c for c in df_analysis.columns if c not in REGION_DUMMY_COLS + STATE_DUMMY_COLS]
print(f"\nBase columns ({len(base_cols)} total, excluding dummies):")
for col in base_cols:
    print(f"  • {col} ({df_analysis[col].dtype})")

print(f"\n--- Log-transformed columns ---")
for col in ['log_population', 'log_income', 'log_total_transfers']:
    print(f"  • {col}")
print(f"\n--- IPCA-adjusted income column ---")
print(f"  • avg_income_real_2022_2022_brl (inflation-adjusted to 2022 BRL)")
print(f"\n--- Sample data (first 5 municipalities, base columns) ---")
display(df_analysis[base_cols].head())

In [ ]:
# ============================================================
# DATASET 6: consolidated_clustering
# Grain: 1 row per municipality (5,565 rows, 5 fewer due to missing data)
# Contains: ML-ready data with BOTH 2010+2022 + z-score normalized features
# ============================================================
df_clustering = datasets.get('consolidated_clustering')

print("=" * 80)
print("DATASET 6: consolidated_clustering (ML-Ready)")
print("=" * 80)
print(f"Shape: {df_clustering.shape}")

print(f"\n--- Raw value columns (both census years) ---")
raw_cols = [c for c in df_clustering.columns if not c.endswith('_norm') and 
            c not in ['municipality_code', 'municipality_name', 'state_code', 'state_abbrev', 
                     'state_name', 'region_code', 'region_name']]
for col in raw_cols:
    print(f"  • {col}")

print(f"\n--- Normalized columns (z-score, for ML) ---")
norm_cols = [c for c in df_clustering.columns if c.endswith('_norm')]
for col in norm_cols:
    print(f"  • {col}")

print(f"\n--- Log columns (for regression) ---")
log_cols_cluster = [c for c in df_clustering.columns if c.startswith('log_')]
for col in log_cols_cluster:
    print(f"  • {col}")

print(f"\n--- Sample data: 2010 vs 2022 comparison ---")
compare_cols = ['municipality_name', 'state_abbrev', 'population_2010', 'population_2022',
                'avg_income_real_2010_2022_brl', 'avg_income_real_2022_2022_brl',
                'income_change_real_pct']
display(df_clustering[compare_cols].head(10))

### Dataset Summary: Which Dataset to Use When?

| Analysis Goal | Recommended Dataset | Why |
|--------------|---------------------|-----|
| **Compare 2010↔2022 census change** | `municipality_socioeconomic` or `consolidated_clustering` | Both contain both census years with inflation adjustment |
| **State-level policy analysis** | `state_summary` | Population-weighted, stable aggregates |
| **Sanctions registry analysis** | `sanctions_summary` | CEIS/CNEP/CEPIM breakdown |
| **Main thesis analysis** | `analysis_compliance_municipality` | **Primary grain** - 5,570 municipalities with transfers + sanctions |
| **Regression/ML modeling** | `consolidated_clustering` | Clean, normalized, no missing values |
| **Compliance trends over time** | `analysis_compliance` (state) or municipality version | State-level for small-N analysis |

### Income & Inflation Reference

**Column naming convention:** `avg_income_real_{value_year}_{base_year}_brl`
- `value_year`: The census year the income was measured
- `base_year`: The inflation base year (2022 = IPCA reference year)

**Examples:**
- `avg_income_real_2022_2022_brl` = 2022 income in 2022 BRL (nominal = real, base year)
- `avg_income_real_2010_2022_brl` = 2010 income restated to 2022 BRL (inflation-adjusted)
- `income_change_real_pct` = Real change % between the two (inflation-adjusted comparison)

In [ ]:
df_analysis.info(verbose=False)

## 3. Descriptive Statistics

In [ ]:
# Describe only the analytical features, excluding the 32 one-hot dummies
# (dummies are summarized separately below).
numeric_cols = df_analysis.select_dtypes(include=[np.number]).columns
analytical_numeric = [c for c in numeric_cols if c not in REGION_DUMMY_COLS + STATE_DUMMY_COLS]
df_analysis[analytical_numeric].describe().T

### Key Metrics Summary

In [ ]:
# Population-weighted averages for rates. A straight mean across 5,570
# municipalities would treat a 500-inhabitant town and Sao Paulo city equally,
# which is statistically misleading for rate and average indicators.
_pop = df_analysis['population_2022']
_lit_mask = df_analysis['literacy_rate_2022'].notna()
_inc_mask = df_analysis['avg_income_2022'].notna()

total_pop = int(_pop.sum())
total_sanc = int(df_analysis['n_sanctions'].sum())

summary = pd.DataFrame({
    'Total Municipalities': [len(df_analysis)],
    'Total States': [int(df_analysis['state_code'].nunique())],
    'Total Regions': [int(df_analysis['region_code'].nunique())],
    'Total Population (2022)': [total_pop],
    'Total Sanctions': [total_sanc],
    'Sanctions/100k (national, pop-weighted)': [round(total_sanc / total_pop * 100_000, 2)],
    'Literacy % (pop-weighted)': [round(np.average(df_analysis.loc[_lit_mask, 'literacy_rate_2022'], weights=_pop[_lit_mask]), 2)],
    'Avg Income BRL (pop-weighted)': [round(np.average(df_analysis.loc[_inc_mask, 'avg_income_2022'], weights=_pop[_inc_mask]), 2)],
})

summary.T

**Dummy variable summary.** For the one-hot region and state dummies the mean equals the proportion of municipalities in that region / state (i.e. `df['is_region_1'].mean()` is the share of munis in the Norte region). A short summary is shown below so we do not flood the main `.describe()` table with 32 extra rows.

In [ ]:
dummy_stats = pd.DataFrame({
    'Count (=1)': df_analysis[REGION_DUMMY_COLS + STATE_DUMMY_COLS].sum(),
    'Share of munis %': (df_analysis[REGION_DUMMY_COLS + STATE_DUMMY_COLS].mean() * 100).round(2),
}).sort_values('Count (=1)', ascending=False)
print(f"Total dummies: {len(dummy_stats)} (5 regions + {len(STATE_DUMMY_COLS)} states)")
dummy_stats.head(10)

## 4. Missing Values Analysis

In [ ]:
missing = df_analysis.isnull().sum()
missing_pct = (missing / len(df_analysis)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct.round(2),
}).sort_values('Missing Count', ascending=False)

missing_df[missing_df['Missing Count'] > 0]

## 5. Distribution Analysis

### 5.1 Target Variable: Sanctions per 100k

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(df_analysis['sanctions_per_100k'], bins=20, edgecolor='black', alpha=0.7)
axes[0].set_title('Distribution of Sanctions per 100k', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Sanctions per 100k Population')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df_analysis['sanctions_per_100k'].mean(), color='red', linestyle='--', label='Mean')
axes[0].axvline(df_analysis['sanctions_per_100k'].median(), color='green', linestyle='--', label='Median')
axes[0].legend()

axes[1].boxplot(df_analysis['sanctions_per_100k'])
axes[1].set_title('Boxplot: Sanctions per 100k', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Sanctions per 100k Population')

from scipy import stats
stats.probplot(df_analysis['sanctions_per_100k'], dist="norm", plot=axes[2])
axes[2].set_title('Q-Q Plot: Sanctions per 100k', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Skewness: {df_analysis['sanctions_per_100k'].skew():.3f}")
print(f"Kurtosis: {df_analysis['sanctions_per_100k'].kurtosis():.3f}")


### 5.2 Socioeconomic Indicators

**Note on Log Transformation:** Population data is typically highly right-skewed — a few states have extremely large populations while most have much smaller values. The logarithmic transformation (log_population) addresses this by:
1. **Normalizing the distribution** — making it more symmetric for valid statistical analysis
2. **Reducing the influence of extreme values** — preventing large populations from disproportionately affecting correlations and regressions
3. **Enabling percentage-based interpretation** — in regression models, changes represent proportional effects

The histograms below compare the raw population distribution (skewed) with the log-transformed version (more normal).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

axes[0, 0].hist(df_analysis['literacy_rate_2022'].dropna(), bins=40, edgecolor='black', alpha=0.7, color='skyblue')
axes[0, 0].set_title('Literacy Rate 2022 Distribution (by municipality)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Literacy Rate (%)')
axes[0, 0].set_ylabel('Frequency')

axes[0, 1].hist(df_analysis['avg_income_2022'].dropna(), bins=40, edgecolor='black', alpha=0.7, color='lightgreen')
axes[0, 1].set_title('Average Income 2022 Distribution (by municipality)', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Average Income (BRL)')
axes[0, 1].set_ylabel('Frequency')

axes[1, 0].hist(df_analysis['population_2022'].dropna(), bins=40, edgecolor='black', alpha=0.7, color='salmon')
axes[1, 0].set_title('Population 2022 Distribution (by municipality)', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Population')
axes[1, 0].set_ylabel('Frequency')

axes[1, 1].hist(df_analysis['log_population'].dropna(), bins=40, edgecolor='black', alpha=0.7, color='plum')
axes[1, 1].set_title('Log(Population) Distribution (by municipality)', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Log(Population)')
axes[1, 1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 6. Regional Analysis

In [ ]:
# Regional aggregation from MUNICIPALITY-level data, using POPULATION-WEIGHTED
# statistics. A simple groupby('region_name').agg('mean') would treat every
# municipality equally, which is statistically misleading: tiny towns would
# dominate the average for rates and monetary values.
# For rates and averages across a region we therefore aggregate from totals
# and weight by municipal population.

def _region_rollup(g: pd.DataFrame) -> pd.Series:
    pop = g['population_2022']
    lit_mask = g['literacy_rate_2022'].notna()
    inc_mask = g['avg_income_2022'].notna()
    return pd.Series({
        'N Municipalities': len(g),
        'N States': g['state_code'].nunique(),
        'Total Population': int(pop.sum()),
        'Total Sanctions': int(g['n_sanctions'].sum()),
        'Sanctions/100k (pop-weighted)': round(g['n_sanctions'].sum() / pop.sum() * 100_000, 2),
        'Literacy % (pop-weighted)': round(np.average(g.loc[lit_mask, 'literacy_rate_2022'], weights=pop[lit_mask]), 2),
        'Avg Income BRL (pop-weighted)': round(np.average(g.loc[inc_mask, 'avg_income_2022'], weights=pop[inc_mask]), 2),
    })

regional_summary = (
    df_analysis.groupby('region_name', observed=True)
    .apply(_region_rollup)
)

regional_summary

In [ ]:
# Regional dashboard built from municipality-level data (pop-weighted rates).
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Sanctions per 100k by Region (pop-weighted)',
                    'Literacy Rate by Region (pop-weighted)',
                    'Average Income by Region (pop-weighted)',
                    'Total Sanctions by Region'),
    specs=[[{'type': 'bar'}, {'type': 'bar'}],
           [{'type': 'bar'}, {'type': 'bar'}]]
)

regions = regional_summary.reset_index()

fig.add_trace(go.Bar(x=regions['region_name'], y=regions['Sanctions/100k (pop-weighted)'],
                     name='Sanctions/100k', marker_color='indianred'), row=1, col=1)
fig.add_trace(go.Bar(x=regions['region_name'], y=regions['Literacy % (pop-weighted)'],
                     name='Literacy %', marker_color='lightseagreen'), row=1, col=2)
fig.add_trace(go.Bar(x=regions['region_name'], y=regions['Avg Income BRL (pop-weighted)'],
                     name='Income', marker_color='lightsalmon'), row=2, col=1)
fig.add_trace(go.Bar(x=regions['region_name'], y=regions['Total Sanctions'],
                     name='Total Sanctions', marker_color='mediumpurple'), row=2, col=2)

fig.update_layout(height=800, showlegend=False, title_text="Regional Comparison Dashboard (built from 5,570 municipalities)")
fig.show()

## 7. State-Level Rollup and Municipality Extremes

We look at both ends of the granularity spectrum:

1. **State-level rollup** (population-weighted from the 5,570 municipalities) for a stable, policy-relevant bar chart.
2. **Top / bottom municipalities** by `sanctions_per_100k`, which can surface individual outliers that the state rollup hides.

In [ ]:
# Top 10 and bottom 10 MUNICIPALITIES by sanctions per 100k.
# NOTE: per-capita rates for very small municipalities can be unstable
# (denominator effect) -- use with care.
cols = ['municipality_code', 'municipality_name', 'state_name', 'region_name',
        'population_2022', 'n_sanctions', 'sanctions_per_100k']

top_10_munis = df_analysis.nlargest(10, 'sanctions_per_100k')[cols]
bottom_10_munis = df_analysis.nsmallest(10, 'sanctions_per_100k')[cols]

print("TOP 10 MUNICIPALITIES - Highest Sanctions per 100k")
print("=" * 90)
print(top_10_munis.to_string(index=False))

print("\n\nBOTTOM 10 MUNICIPALITIES - Lowest Sanctions per 100k (among munis with sanctions > 0)")
print("=" * 90)
with_sanctions = df_analysis[df_analysis['n_sanctions'] > 0]
print(with_sanctions.nsmallest(10, 'sanctions_per_100k')[cols].to_string(index=False))

In [ ]:
# State-level rollup from municipality data (population-weighted).
# state_name is used on the x-axis (qualitative label); state_code is just an id.
state_rollup = (
    df_analysis.groupby(['state_code', 'state_name', 'region_name'], observed=True)
    .apply(lambda g: pd.Series({
        'population': g['population_2022'].sum(),
        'n_sanctions': g['n_sanctions'].sum(),
        'sanctions_per_100k': g['n_sanctions'].sum() / g['population_2022'].sum() * 100_000,
    }))
    .reset_index()
    .sort_values('sanctions_per_100k', ascending=False)
)

fig = px.bar(state_rollup,
             x='state_name', y='sanctions_per_100k',
             color='region_name',
             title='Sanctions per 100k Population by State (rolled up from 5,570 municipalities)',
             labels={'sanctions_per_100k': 'Sanctions per 100k', 'state_name': 'State'},
             height=500)
fig.update_xaxes(tickangle=-45)
fig.show()

## 8. Sanctions Registry Analysis

In [ ]:
if df_sanctions is not None:
    print("Sanctions by Registry Type:")
    print("=" * 60)
    display(df_sanctions[['registry_type', 'total_sanctions', 'sanctions_pf', 'sanctions_pj', 'pj_ratio_pct']])
    
    fig = px.pie(df_sanctions, values='total_sanctions', names='registry_type',
                 title='Sanctions Distribution by Registry Type')
    fig.show()


## 9. Correlation Heatmap (Preview)

In [ ]:
# Correlation matrix of key analytical features at municipality level
# (5,570 observations instead of 27 states -- much more statistical power).
# We include the region dummies but NOT the 27 state dummies (would make the
# heatmap unreadable). Transfer-side features are also included since the
# thesis question links federal transfers to compliance outcomes.

corr_cols = ['sanctions_per_100k', 'literacy_rate_2022', 'avg_income_2022',
             'log_population', 'log_income']
# Include log_total_transfers if present (new in muni dataset)
if 'log_total_transfers' in df_analysis.columns:
    corr_cols.append('log_total_transfers')
corr_cols += REGION_DUMMY_COLS

# Cast to float64 (numpy) so np.corrcoef / seaborn are happy with pandas
# nullable Int64/Float64 + rows that contain NaN in any included column.
corr_df = df_analysis[corr_cols].astype('Float64').astype(float)
corr_matrix = corr_df.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix: Key Variables (municipality-level, N=5,570)',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Key Findings Summary

### 10.1 Data Quality
- **Grain:** this EDA runs at **municipality level** (5,570 rows, 1 per Brazilian municipality), up from the previous state-level view (27 rows). This gives ~200x more statistical power for correlation and regression work downstream.
- **Complete geographic coverage**: all 27 states and all 5 regions are represented.
- **Sanctions data is dense**: `n_sanctions` is non-null for all 5,570 municipalities (zeros are real, not missing).
- **Transfer-rate feature is sparse**: `sanctions_per_million_brl_transfers` is null for ~91.5% of municipalities (most have no federal transfer records in the current Gold cut). Use with care in any modeling that depends on it.

### 10.2 Regional Patterns (population-weighted)
- Regional numbers are computed as `sum(sanctions) / sum(population) * 100_000` across each region's municipalities -- not as an unweighted mean of per-muni rates -- so they are not dominated by tiny municipalities.
- Once weighted correctly, the ordering of regions by sanctions/100k is flatter than the old state-level unweighted view suggested; see the regional summary table above for the actual values on the current Gold snapshot.

### 10.3 State and Municipality Extremes
- The state-level bar chart is now computed as a **rollup from municipality data** (population-weighted), so every state's number is consistent with the regional totals.
- The top-10 municipalities by `sanctions_per_100k` surface individual outliers that the state rollup hides. Rates for very small municipalities can be unstable (denominator effect) and should be interpreted alongside absolute `n_sanctions` and `population_2022`.

### 10.4 Dummy variables
- Region and state are preserved as identifier columns (`state_code`, `state_name`, `region_code`, `region_name`) AND as one-hot dummies (`is_region_*`, `is_state_*`) for use as regression features.
- The mean of a dummy equals the proportion of municipalities in that category (e.g. `df['is_region_1'].mean()` == share of munis in the Norte region). See the dummy summary table for the actual shares.

### 10.5 Correlations (municipality-level)
- With N=5,570 the correlation coefficients in the heatmap are far more reliable than at the 27-state grain. Inspect the `sanctions_per_100k` row / column in the heatmap above for the strongest bivariate signals; the thesis question about federal transfers vs. compliance outcomes can now be tested at the unit of observation where policy actually lands (the municipality).